# Phase 6: the LLM counterfactual arm (Colab)

The decisive comparison. Both arms get the **same fixed regions** and the
same condition prompts; only the colour source differs:

- **KB arm** — deterministic, computed locally by `apply_condition()`
- **LLM arm** — Qwen2.5-VL picks a hex colour per region, told the condition

**Pre-registered prediction:** the LLM shows *weaker separation* between
conditions than the KB — it has no systematic period/mood mapping, and its
documented gray-hedging (Phase 5, p=7e-05) leaves it nowhere to move.
If separation is near zero, the LLM is effectively ignoring the context
prompt, which is the whole argument for an explicit KB.

> Runtime → GPU. ~4 conditions x N images LLM calls; budget ~20-30 min for N=23.

In [ ]:
!nvidia-smi -L
%cd /content
!test -d chroma-reasoner || git clone https://github.com/tomqi6195/chroma-reasoner.git
!cd chroma-reasoner && git pull && git log --oneline -1
!pip install -q -e chroma-reasoner
!pip install -q "transformers<5" accelerate pycocotools
import sys; sys.path.insert(0, '/content/chroma-reasoner/src')

## 1. Source plans

Uses `plans/reasoned/` from the clone. If that only holds the older 5-image
set, upload a zip of your local `plans/reasoned` when prompted (or commit
the 23-image set and re-run the cell above).

In [ ]:
import glob, json, os, shutil, urllib.request
import cv2

os.makedirs('plans/reasoned', exist_ok=True)
for src in glob.glob('chroma-reasoner/plans/reasoned/*.json'):
    shutil.copy(src, 'plans/reasoned/')
print('plans from repo:', len(glob.glob('plans/reasoned/*.json')))

UPLOAD_MORE = False   # set True to upload a zip of your local plans/reasoned
if UPLOAD_MORE:
    from google.colab import files
    up = files.upload()
    for name in up:
        !unzip -oq {name}
    print('plans now:', len(glob.glob('plans/reasoned/*.json')))

In [ ]:
# Grayscale inputs for the images the plans refer to
os.makedirs('gray', exist_ok=True)
IMAGE_IDS = [os.path.basename(p)[:-5] for p in sorted(glob.glob('plans/reasoned/*.json'))]
for iid in IMAGE_IDS:
    if os.path.exists(f'gray/{iid}.png'):
        continue
    dest = f'/tmp/{iid}.jpg'
    urllib.request.urlretrieve(f'http://images.cocodataset.org/val2017/{iid}.jpg', dest)
    img = cv2.imread(dest)
    cv2.imwrite(f'gray/{iid}.png', cv2.cvtColor(img, cv2.COLOR_BGR2LAB)[:, :, 0])
print(len(IMAGE_IDS), 'images ready')

## 2. KB arm (local, deterministic — no model)

In [ ]:
from chroma_reasoner.eval.counterfactual import (CONDITIONS, condition_variants,
                                                 evaluate_all, format_contrast)
from chroma_reasoner.kb import load_kb
from chroma_reasoner.plan import load_plan

# The contrast pairs we score; trim to cut LLM cost.
USE_CONDITIONS = ['mood_melancholic', 'mood_cheerful', 'era_1910s', 'era_1970s']

kb = load_kb('/content/chroma-reasoner/kb')
kb_arm = {name: {} for name in USE_CONDITIONS}
for plan_path in sorted(glob.glob('plans/reasoned/*.json')):
    plan = load_plan(plan_path)
    for name, variant in condition_variants(kb, plan, USE_CONDITIONS).items():
        kb_arm[name][plan['image_id']] = variant
        out_dir = f'plans/counterfactual_kb/{name}'
        os.makedirs(out_dir, exist_ok=True)
        with open(f"{out_dir}/{plan['image_id']}.json", 'w') as f:
            json.dump(variant, f, indent=2)
print('KB arm ready:', {k: len(v) for k, v in kb_arm.items()})

## 3. LLM arm (same regions, same prompts, model picks colours)

In [ ]:
from chroma_reasoner.eval import llm_color_plan
from chroma_reasoner.reasoner.backend_open import QwenVLBackend

def log(*parts):
    line = ' '.join(str(p) for p in parts)
    print(line)
    with open('phase6_log.txt', 'a') as lf:
        lf.write(line + '\n')

backend = QwenVLBackend(model_id='Qwen/Qwen2.5-VL-7B-Instruct')
print('backend ready')

In [ ]:
# The condition's prompt rides on plan['prompt'], which llm_color_plan puts
# in front of the model — so the LLM is told exactly what the KB was told.
llm_arm = {name: {} for name in USE_CONDITIONS}
total = sum(len(v) for v in kb_arm.values())
done = 0
for name in USE_CONDITIONS:
    for iid, cond_plan in kb_arm[name].items():
        done += 1
        try:
            out = llm_color_plan(cond_plan, backend, f'gray/{iid}.png')
        except Exception as e:
            log(f'!! {name}/{iid}: {type(e).__name__}: {e}'); continue
        llm_arm[name][iid] = out
        out_dir = f'plans/counterfactual_llm/{name}'
        os.makedirs(out_dir, exist_ok=True)
        with open(f'{out_dir}/{iid}.json', 'w') as f:
            json.dump(out, f, indent=2)
        if done % 10 == 0:
            log(f'  {done}/{total}')
log('LLM arm:', {k: len(v) for k, v in llm_arm.items()})

## 4. The comparison: does either arm respond to context?

In [ ]:
kb_res = evaluate_all(kb_arm)
llm_res = evaluate_all(llm_arm)

log('\n================ KB ARM ================')
for r in kb_res:
    log(format_contrast(r))
log('\n================ LLM ARM ===============')
for r in llm_res:
    log(format_contrast(r))

log('\n============ SEPARATION HEAD-TO-HEAD ===========')
log(f"{'contrast':>34} {'kb sep':>8} {'llm sep':>8} {'kb active':>10} {'llm active':>11}")
by_pair = {(r['a'], r['b']): r for r in llm_res}
for r in kb_res:
    l = by_pair.get((r['a'], r['b']))
    if not l:
        continue
    log(f"{r['a'] + ' vs ' + r['b']:>34} "
        f"{r['median_separation']:>8} {l['median_separation']:>8} "
        f"{r['active_share']:>10} {l['active_share']:>11}")

with open('phase6_results.json', 'w') as f:
    json.dump({'kb': kb_res, 'llm': llm_res}, f, indent=2)

In [ ]:
!zip -rq phase6_outputs.zip plans/counterfactual_kb plans/counterfactual_llm phase6_results.json phase6_log.txt
from google.colab import files
files.download('phase6_outputs.zip')